# Ch. 2 · TP1 Exploration des données scientifiques

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch02-tp1-exploration.ipynb)

**IA pour les sciences — M1.** Décrire, résumer et représenter des mesures avec pandas et Matplotlib.

**Durée indicative :** 3 h

**Objectifs**

- Construire un tableau de mesures avec identifiants, contexte et unités.
- Calculer des statistiques descriptives globales et par groupe.
- Produire des figures scientifiques correctes (axes, unités, titre, effectif).
- Montrer, avec le quartet d’Anscombe, qu’un résumé chiffré ne suffit pas.

**Prérequis :** chapitre 1 ; bases de Python (variables, listes, boucles).

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Construire un jeu de mesures traçable

Nous simulons une campagne de 60 prélèvements d’eau sur **trois sites**. Chaque ligne a un identifiant,
un site, une température (°C) et une concentration (mg/L). La relation simulée (0,4 mg/L par °C, plus un effet de site)
sert uniquement à l’exercice : **ce n’est pas une loi physique**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 60
sites = rng.choice(['Amont', 'Centre', 'Aval'], size=n)
effet_site = pd.Series(sites).map({'Amont': 0.0, 'Centre': 1.5, 'Aval': 3.0}).to_numpy()
df = pd.DataFrame({
    'echantillon': [f'E{i:03d}' for i in range(n)],
    'site': sites,
    'temperature_C': rng.uniform(15, 35, n).round(1),
})
df['concentration_mg_L'] = (0.4 * df['temperature_C'] + effet_site
                            + rng.normal(0, 2, n)).round(2)
print(df.head())
print(df.dtypes)

**🔎 À observer.** `dtypes` indique le type informatique : `site` est catégoriel (`object`), les mesures sont numériques.
L’identifiant `echantillon` n’est **pas** une variable explicative : il ne doit jamais entrer dans un modèle.

## 2. Statistiques descriptives

- **Moyenne** $\bar{x} = \frac{1}{n}\sum x_i$ : sensible aux valeurs extrêmes.
- **Médiane** : valeur centrale, robuste.
- **Écart-type** $s = \sqrt{\frac{1}{n-1}\sum (x_i-\bar{x})^2}$ : dispersion autour de la moyenne.
- **Quartiles** $Q_1, Q_3$ et écart interquartile $IQR = Q_3 - Q_1$.

In [ ]:
num = ['temperature_C', 'concentration_mg_L']
print(df[num].describe().round(2))
print('\nValeurs manquantes :\n', df.isna().sum())
print('\nCorrélation de Pearson :\n', df[num].corr().round(3))

## 3. Comparer des groupes : `groupby`

En sciences, une moyenne globale peut masquer des différences entre sites, lots ou instruments.

In [ ]:
resume = (df.groupby('site')['concentration_mg_L']
            .agg(['count', 'mean', 'median', 'std'])
            .round(2))
print(resume)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column='concentration_mg_L', by='site', ax=axes[0])
axes[0].set(title='Concentration par site', xlabel='Site', ylabel='Concentration (mg/L)')
for site, g in df.groupby('site'):
    axes[1].scatter(g['temperature_C'], g['concentration_mg_L'], label=f'{site} (n={len(g)})', alpha=.8)
axes[1].set(xlabel='Température (°C)', ylabel='Concentration (mg/L)',
            title='Concentration selon la température')
axes[1].legend()
plt.suptitle('Données simulées — exercice pédagogique')
plt.tight_layout(); plt.show()

## 4. Démonstration : le quartet d’Anscombe

Anscombe (1973) a construit quatre jeux de 11 points qui ont **la même moyenne, la même variance, la même corrélation
et la même droite de régression**… et des formes complètement différentes. Moralité : **toujours visualiser**.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    'I':   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    'II':  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    'III': (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    'IV':  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
            [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (nom, (xs, ys)) in zip(axes, anscombe.items()):
    xs, ys = np.array(xs), np.array(ys)
    pente, origine = np.polyfit(xs, ys, 1)
    r = np.corrcoef(xs, ys)[0, 1]
    ax.scatter(xs, ys)
    grille = np.linspace(3, 20, 10)
    ax.plot(grille, origine + pente * grille, color='C1')
    ax.set_title(f'{nom} : moy y={ys.mean():.2f}, r={r:.3f}\ny = {origine:.2f} + {pente:.2f}x', fontsize=9)
plt.tight_layout(); plt.show()

### ✏️ Exercice 1 — Histogramme correct

Tracez l’histogramme de la température avec 12 classes, un titre informatif, des axes avec unités et l’effectif $n$ dans le titre.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(..., bins=..., edgecolor='white')
ax.set(xlabel=..., ylabel=..., title=...)
plt.show()

<details><summary><b>Voir le corrigé</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['temperature_C'], bins=12, edgecolor='white')
ax.set(xlabel='Température (°C)', ylabel='Nombre d’échantillons',
       title=f'Distribution des températures (n = {len(df)}, données simulées)')
plt.show()
```

</details>

### ✏️ Exercice 2 — Conversion d’unités et traçabilité

Ajoutez une colonne `concentration_umol_L` en supposant une masse molaire de 62 g/mol (1 mg/L = 1000/62 µmol/L). Vérifiez ensuite que la corrélation avec la température est inchangée. Pourquoi ?

In [ ]:
M = 62.0  # g/mol
df['concentration_umol_L'] = ...
print(df[['temperature_C', 'concentration_mg_L', 'concentration_umol_L']].corr().round(3))

<details><summary><b>Voir le corrigé</b></summary>

```python
M = 62.0  # g/mol
df['concentration_umol_L'] = df['concentration_mg_L'] * 1000 / M
print(df[['temperature_C', 'concentration_mg_L', 'concentration_umol_L']].corr().round(3))
```

Une conversion linéaire $x \mapsto a x$ ($a>0$) ne change pas la corrélation de Pearson. En revanche, elle change moyennes, écarts-types et coefficients d’un modèle : l’unité doit toujours être documentée.

</details>

### ✏️ Exercice 3 — Statistiques par site

Pour chaque site, calculez la pente de la droite concentration = f(température) avec `np.polyfit`. Les pentes sont-elles proches de la valeur simulée 0,4 ? Pourquoi ne sont-elles pas exactement égales ?

In [ ]:
for site, g in df.groupby('site'):
    pente, origine = ...
    print(site, len(g), round(pente, 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
for site, g in df.groupby('site'):
    pente, origine = np.polyfit(g['temperature_C'], g['concentration_mg_L'], 1)
    print(site, len(g), round(pente, 3))
```

Chaque pente est estimée sur ~20 points bruités : elle fluctue autour de 0,4. L’écart entre sites ne prouve donc pas une différence de mécanisme ; il faudrait quantifier l’incertitude de chaque pente.

</details>

In [ ]:
df.to_csv('mesures_fictives.csv', index=False)
print('Fichier écrit : mesures_fictives.csv', df.shape)

## Interprétation scientifique (à rédiger)

1. Décrivez les données (n, sites, unités, plage de température).
2. La concentration diffère-t-elle selon le site ? Sur quelle figure vous appuyez-vous ?
3. Que montre le quartet d’Anscombe sur l’usage des seuls résumés numériques ?
4. Quelles informations de contexte manquent pour interpréter ces mesures réelles ?